# <center>RumbleDB sandbox</center>


This is a RumbleDB notebook that allows you to play with simple XQuery queries with the RumbleDB engine.

It is a jupyter notebook that you can also download and execute on your own machine, but if you arrived here from the RumbleDB website, it is likely to be shown within Google's Colab environment.

To get started, we first need to make sure Java 17 (or 21) is installed. If the output below says 17, then nothing needs to be done.

Otherwise, the commented lines were prepared to install Java when using Google Colab (or for Ubuntu generally), which is the setup that jsoniq.org and rumbledb.org link to. If you downloaded the notebook and have another Operating System than Linux, you need to use the Java install command corresponding to your Operating System for this to work.


In [7]:
!java -version
# We make sure Java 17 is installed.
#!apt update
#!apt install openjdk-17-jdk


openjdk version "17.0.13" 2024-10-15
OpenJDK Runtime Environment Temurin-17.0.13+11 (build 17.0.13+11)
OpenJDK 64-Bit Server VM Temurin-17.0.13+11 (build 17.0.13+11, mixed mode, sharing)


Now we install the jsoniq Python library.

In [8]:
#%pip install jsoniq

Here, the JSONiq queries are executed locally to the notebook. Advanced users can also run a large Spark cluster and execute JSONiq queries on Petabytes of data.

JSONiq queries can generally be evaluated in Python as follows. It is possible to provide as input Python dicts, lists, pandas DataFrames, and it is possible to retrieve the results as Python values, pandas DataFrames, etc.

In [9]:
from jsoniq import RumbleSession

rumble = RumbleSession.builder.getOrCreate();

print(rumble.xquery(r"""
    <a>{map { "foo": [ 6*7 ] }?foo}</a>
""").serialize())

<?xml version="1.0" encoding="UTF-8"?><a>42</a>


However, for convenience, we activate the jsoniq magic. This means you can directly type the XQuery queries in a notebook cell, without needing to wrap them inside rumble.jsoniq() calls. This makes it easier to read.

In [37]:
%load_ext jsoniqmagic


The jsoniqmagic extension is already loaded. To reload it, use:
  %reload_ext jsoniqmagic


In [42]:
%%xquery
<a>{
    for $x in 1 to 10
    return <b>{map { "foo": [ 6*7 ] }?foo}</b>
}</a>

<?xml version="1.0" encoding="UTF-8"?><a><b>42</b><b>42</b><b>42</b><b>42</b><b>42</b><b>42</b><b>42</b><b>42</b><b>42</b><b>42</b></a>


## Serialization

RumbleDB fully implements the XSLT and XQuery Serialization 3.1 specification.


In [43]:
%%xquery

declare namespace output =
  "http://www.w3.org/2010/xslt-xquery-serialization";

declare option output:method "xml";
declare option output:indent "yes";
declare option output:omit-xml-declaration "yes";

<a>{
    for $x in 1 to 10
    return <b>{map { "foo": [ 6*7 ] }?foo}</b>
}</a>


<a>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
  <b>42</b>
</a>


In [49]:
%%xquery

declare namespace output =
  "http://www.w3.org/2010/xslt-xquery-serialization";

declare option output:method "json";
declare option output:indent "no";
declare option output:omit-xml-declaration "yes";

array {
    for $x in 1 to 10
    return map { "foo": [ 6*7 ] }
}


[{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]},{"foo":[42]}]


## Leveraging RumbleDB's scalability
RumbleDB is very powerful and can run on multiple cores as well as clusters of matchines in a data center.
You can also create structured output with sequence of maps that are fully interoperable with Python's Pandas library. 

In [62]:
%%xquery -pdf
for $x in 1 to 10
return map { "n" : $x, "s" : math:pow($x, 2) }

,n,s
0,1,1.0
1,2,4.0
2,3,9.0
3,4,16.0
4,5,25.0
5,6,36.0
6,7,49.0
7,8,64.0
8,9,81.0
9,10,100.0


In [68]:
from jsoniq import RumbleSession

rumble = RumbleSession.builder.getOrCreate();

pandas = rumble.xquery(r"""
    for $x in 1 to 10
    return map { "n" : $x, "s" : math:pow($x, 2) }
""").pdf()

pandas = (
    pandas.loc[pandas["n"] % 2 == 0]
    .assign(cube=lambda d: d["n"] ** 3)
    .sort_values("s", ascending=False)
    .reset_index(drop=True)
)

print(pandas)
print("Sum of squares:", pandas["s"].sum())

    n      s  cube
0  10  100.0  1000
1   8   64.0   512
2   6   36.0   216
3   4   16.0    64
4   2    4.0     8
Sum of squares: 220.0
